This is a set of notes about logistic regression. 
### Math
- This is for binary classification
- We still have our input multiplied by the weight and then we add the bias $Wx+b$ but now we use a sigmoid function to transform this result to probabilites.
  - The sigmoid function is $\sigma (z) = \frac{1}{1+e^{-z}}$ which maps $\mathbb{R} \to (0,1)$. Our input will be the linear combination we got previously $z = Wx+b$ 
- We are trying to maximize the likelihood function (or minimize the negative likelihood) which is $L(\cdot)= \Pi \space \sigma (z_i)^{y_i} \cdot (1-\sigma (z_i))^{1-y_i}$
      - The likelihood has a nice property that when $y_i=1$ the second term equals to one and the first term is maximized when $\sigma(z_i)$ is close to one. Works similarly for when $y_i =0$ 
  - We take the log of both sides since maximizing the log is the same as maximizing the likelihood: $log(L()) = \sum y_i log(\sigma (z_i))+(1-y_i)log(1-\sigma (z_i))$
  - The gradient turns out to be the same as in linear regression, so $\frac{\partial log(L(\cdot))}{\partial w} = x_i (\hat{y}_i - y_i)$ and $\frac{\partial log(L(\cdot))}{\partial b}= (\hat{y}_i-y_i)$ 
- Then using gradient descent we update the weights.
- Then we use a threshold value (e.g. $\hat{y} \geq 0.5$) to determine the prediction at inference time 

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
class LogisticRegression():
    def __init__(self, num_of_features):
        self.w = np.random.randn(num_of_features,1)
        self.b = 0
        
    def predict(self, X):
        z = X @ self.w + self.b
        z = np.clip(z,-300,300)    # this is to prevent some numerical instability that happens at extreme values of z
        return 1 / (1 + np.exp(-z))
        
    def train(self, X, Y, a, epochs):
        
        for _ in range(epochs):
            error = self.predict(X) - Y 
            gradient_w = X.T @ error / len(Y)
            gradient_b = np.mean(error) 
            
            self.w -= a * gradient_w
            self.b -= a * gradient_b

In [ ]:
np.random.seed(42)

N = 100
X = np.arange(N).reshape(-1,1)
Y = np.where((X>29), 0, 1).reshape(-1,1)
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2)

model = LogisticRegression(1)
model.train(X_train, Y_train, 0.01, 10000)

predictions = model.predict(X)

plt.plot(Y)
plt.plot(predictions)

While this works well with simple boundary lines, it fails with more complicated examples. What the model is currently doing is just fitting
a sigmoid function onto the curve to minimize mistakes, but that doesn't always work well.

In [ ]:
X = np.arange(100).reshape(-1,1)
Y = np.where((X>29) & (X < 60), 0, 1).reshape(-1,1)
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2)


model = LogisticRegression(1)
model.train(X_train, Y_train, 0.01, 10000)

predictions = model.predict(X)

plt.plot(Y)
plt.plot(predictions)

This vanilla logistic regression is a **linear classifier** meaning that it just finds a boundary line ($x=c$) and outpus one class beyond the line and another before the line.
This line corresponds to where $\sigma (z)=0.5$ equivalently $Wx+b=0$ since $\sigma(0)=0.5$. If our features are linear so is the boundary. 
We can use feature engineering to create a non-linear boundary. 
This is like finding roots ($x=c$) such that the boundary equation is 0, I wonder what the intuition is here as we have more complex polynomials with more roots.

In [ ]:
X = np.arange(100).reshape(-1,1)
Y = np.where((X>29) & (X < 60), 0, 1).reshape(-1,1)
X = np.hstack((X,X**2))
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2)

model = LogisticRegression(2)
model.train(X_train, Y_train, 0.0001, 10000)

predictions = model.predict(X)

plt.plot(Y)
plt.plot(predictions)

So the implementation is fine, but the results are pretty bad.
This is likely due to problems with the scale of our engineered column. It is up to 100x bigger than the original.
Due to that, $Wx+b$ will result in huge $z$ which will make the prediction 1 pretty much most of the time, and changing the $w$ by a little bit with
such large scales won't really affect the prediction that much so intuitively the gradient vanishes and the model doesn't ever correct itself.
Also the scale of the $X^2$ causes the rates of learning to be different so picking a proper $\alpha$ becomes impossible.

In [ ]:
X = (X - np.mean(X,axis=0)) / np.std(X,axis=0)
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2)

model = LogisticRegression(2)
model.train(X_train, Y_train, 0.1, 10000)

predictions = model.predict(X)

plt.plot(Y)
plt.plot(predictions)

This all works fine and well when we have two classes, but sometimes we will have more to work with. This is where we use multinomial logistic regression
with the softmax function instead of the sigmoid function. 
The softmax function is: $$\sigma(z)_i = \frac{e^{z_i}}{\sum_{j=1}^K e^{z_j}} $$ 
- The function outputs a vector where all the elements represent the probabilities that an input is in that class.
   - The elements of the vector sum up to 1. 
- This is also prone to numerical instability so one solution is to subtract the maximum $z$ value from the exponents i.e. $$\sigma(z)_i = \frac{e^{z_i-max(z)}}{\sum_{j=1}^K e^{z_j-max(z)}} $$ (https://jaykmody.com/blog/stable-softmax/). I found clipping easier to implement but this seemed interesting enough to mention

In [ ]:
class MLTLogisticRegression(LogisticRegression):
    def __init__(self,num_of_features,num_of_classes):
        self.w = np.random.randn(num_of_features,num_of_classes)
        self.b = np.random.randn(1,num_of_classes)
        
    def predict(self, X):
       z = X @ self.w + self.b 
       z = np.clip(z,-300,300)
       return np.exp(z) / (np.sum(np.exp(z), axis=1).reshape(-1,1))

In [ ]:
np.random.seed(42)
num_of_classes = 3
num_of_features = 2
N = 100

X = np.random.rand(N * num_of_features).reshape(N,num_of_features) * 200
Y = np.zeros(N * num_of_classes).reshape(N,num_of_classes)

sum_of_columns = X[:,0] + X[:,1]
Y[:,0] = np.where((sum_of_columns  < 100),1,0)
Y[:,1] = np.where((100 <= sum_of_columns) & (sum_of_columns  < 200),1,0)
Y[:,2] = np.where((200 <= sum_of_columns),1,0)

plt.scatter(X[:,0],X[:,1], c=Y)
plt.title("Data")

In [ ]:
X = (X - np.mean(X,axis=0)) / np.std(X,axis=0)
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2, random_state = 42)

model = MLTLogisticRegression(num_of_features,num_of_classes)
model.train(X_train,Y_train,0.01,30000)
predictions = np.argmax(model.predict(X), axis=1)

plt.scatter(X[:,0],X[:,1], c=predictions, cmap="brg")
plt.title("Model Predictions")